# Athena: Ethiopia El Niño report

Ingest `train_data/impact_report_examples/ethiopia.md` with the Athena agents, then inspect the graph and ask why Ethiopia is exposed to El Niño.

The kernel writes the authoritative graph to the local Fuseki dataset (`http://127.0.0.1:3030/ds`). Batches are drained in this process, and retrieval uses an in-memory index, so RabbitMQ and Atlas are not required for this single document. Extraction, verification, discovery, and research call the Bedrock models from `.env`.

Start Fuseki first (`docker compose up -d fuseki`).


In [1]:
from __future__ import annotations

import logging
import os
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
os.chdir(ROOT)
load_dotenv(ROOT / ".env")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s [%(name)s] %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
for _name in ("httpx", "httpcore", "openai", "botocore", "urllib3"):
    logging.getLogger(_name).setLevel(logging.WARNING)

print("Working directory:", ROOT)


Working directory: /Users/olokshyn/projects/fao/fao_impact_monitor


In [2]:
from fao_impact_monitor.athena.agents.discovery import SdkRelationProposer
from fao_impact_monitor.athena.agents.external import DisabledExternalRetriever
from fao_impact_monitor.athena.agents.ingestion import SdkClaimExtractor
from fao_impact_monitor.athena.agents.ontology import SdkOntologyProposer
from fao_impact_monitor.athena.agents.research import SdkResearchReasoner
from fao_impact_monitor.athena.agents.verification import SdkEvidenceVerifier
from fao_impact_monitor.athena.config import get_athena_config, policies_from_config
from fao_impact_monitor.athena.context import AthenaContext
from fao_impact_monitor.athena.jobs.batch import BatchGate, InProcessBroker
from fao_impact_monitor.athena.model.filters import FilterSpec
from fao_impact_monitor.athena.model.records import ConceptRecord, SourceRecord
from fao_impact_monitor.athena.ns import term
from fao_impact_monitor.athena.observe import MemoryDecisionLog
from fao_impact_monitor.athena.pipeline.handlers import process_batch
from fao_impact_monitor.athena.pipeline.ingest import StoredDocument, ingest_document
from fao_impact_monitor.athena.pipeline.load import (
    load_assertions,
    load_claims,
    load_evidence,
    load_models,
)
from fao_impact_monitor.athena.pipeline.research import run_research
from fao_impact_monitor.athena.storage.graph import FusekiGraph, ensure_ontology
from fao_impact_monitor.athena.storage.index import BedrockEmbedder, MemoryIndex
from fao_impact_monitor.athena.storage.objects import LocalObjectStore
from fao_impact_monitor.athena.storage.rdf import source_iri

settings = get_athena_config().model_copy(
    update={
        "ingest_batch_size": 1,
        "verify_batch_size": 1,
        "discover_batch_size": 1,
        "backfill_batch_size": 1,
        "index_batch_size": 1,
        "maintain_batch_size": 1,
        "research_batch_size": 1,
        "ingest_batch_wait_seconds": 0,
        "verify_batch_wait_seconds": 0,
        "discover_batch_wait_seconds": 0,
        "backfill_batch_wait_seconds": 0,
        "index_batch_wait_seconds": 0,
        "maintain_batch_wait_seconds": 0,
        "research_batch_wait_seconds": 0,
        "research_max_rounds": 2,
    }
)
decisions = MemoryDecisionLog()
broker = InProcessBroker()
graph = FusekiGraph(settings.fuseki_query_url, settings.fuseki_update_url)
ensure_ontology(graph)
ctx = AthenaContext(
    config=settings,
    graph=graph,
    objects=LocalObjectStore(settings.object_store_root),
    index=MemoryIndex(),
    decisions=decisions,
    embedder=BedrockEmbedder(),
    extractor=SdkClaimExtractor(settings.ingestion_model, decisions),
    verifier=SdkEvidenceVerifier(settings.verifier_model, decisions),
    discoverer=SdkRelationProposer(settings.discovery_model, decisions),
    ontologist=SdkOntologyProposer(settings.ontology_model, decisions),
    researcher=SdkResearchReasoner(settings.research_model, decisions),
    external=DisabledExternalRetriever(),
    broker=broker,
    gate=BatchGate(graph, broker, policies_from_config(settings)),
)
print("Fuseki:", settings.fuseki_query_url)
print("Ingestion model:", settings.ingestion_model)
print("Research model:", settings.research_model)


Fuseki: http://127.0.0.1:3030/ds/query
Ingestion model: openai.gpt-6-luna
Research model: openai.gpt-6.1-terra


In [3]:
async def drain() -> None:
    """Run committed batches here. Index updates run before discovery."""
    seen: set[str] = set()
    while True:
        ctx.gate.flush_ready()
        pending = [
            (task_type, batch_id)
            for task_type, batch_id in broker.published
            if batch_id not in seen
        ]
        if not pending:
            return
        pending.sort(key=lambda item: 0 if item[0] == "index_sync" else 1)
        for task_type, batch_id in pending:
            seen.add(batch_id)
            print(f"batch {task_type}")
            try:
                await process_batch(ctx, batch_id)
            except Exception as exc:
                print(f"batch {task_type} failed: {exc}")


report_path = ROOT / "train_data" / "impact_report_examples" / "ethiopia.md"
payload = report_path.read_bytes()
ref = ctx.objects.put(
    "reports/ethiopia-el-nino",
    payload,
    media_type="text/markdown",
)
document = StoredDocument(
    logical_id=ref.logical_id,
    version=ref.version,
    sha256=ref.sha256,
    media_type=ref.media_type,
    title="Ethiopia: El Niño Risk Outlook",
    external_id="ethiopia.md",
    tags={
        "country": ["ETH"],
        "source_type": ["report"],
        "acquisition_channel": ["direct-upload"],
    },
)
claim_ids = await ingest_document(ctx, document)
await drain()
print(f"Ingest finished with {len(claim_ids)} claim ids")


ModelBehaviorError: Invalid JSON when parsing {
  "evidence": [
    {
      "excerpt": "Agriculture, forestry and fishing accounted for about 32.8% of GDP in 2025 [1]",
      "page": null,
      "section": "Past impacts",
      "table_cell": null,
      "start_offset": null,
      "end_offset": null
    },
    {
      "excerpt": "agriculture employed about 60.1% of the workforce [2]",
      "page": null,
      "section": "Past impacts",
      "table_cell": null,
      "start_offset": null,
      "end_offset": null
    },
    {
      "excerpt": "land equipped for irrigation representing only about 4.7% of cropland in 2024 [3]",
      "page": null,
      "section": "Past impacts",
      "table_cell": null,
      "start_offset": null,
      "end_offset": null
    },
    {
      "excerpt": "In western and northern Ethiopia, El Niño is typically associated with below-average June–September rainfall, threatening the main cropping season",
      "page": null,
      "section": "Past impacts",


In [ ]:
sid = source_iri(ref.sha256)
evidence = [
    item for item in load_evidence(ctx.graph).values() if item.source_iri == sid
]
evidence_ids = {item.iri for item in evidence}
claims = sorted(
    (
        item
        for item in load_claims(ctx.graph).values()
        if evidence_ids.intersection(item.evidence_iris)
    ),
    key=lambda item: item.normalized,
)
claim_iris = {item.iri for item in claims}
assertions = sorted(
    (
        item
        for item in load_assertions(ctx.graph).values()
        if item.subject_iri in claim_iris or item.object_iri in claim_iris
    ),
    key=lambda item: (
        item.relation_iri,
        item.subject_iri,
        item.object_iri,
    ),
)
concepts = load_models(ctx.graph, term("Concept"), ConceptRecord)
sources = load_models(ctx.graph, term("Source"), SourceRecord)
used_concepts = {
    item.concept_iri for item in claims if item.concept_iri in concepts
}
node_counts = {
    "Source": int(sid in sources),
    "Evidence": len(evidence),
    "Claim": len(claims),
    "Concept": len(used_concepts),
    "RelationAssertion": len(assertions),
}
relation_types = Counter(item.relation_iri.rsplit("/", 1)[-1] for item in assertions)

print("1. Nodes created:", sum(node_counts.values()))
for name, count in node_counts.items():
    print(f"   {name}: {count}")
print("2. Relation types:")
if relation_types:
    for name, count in sorted(relation_types.items()):
        print(f"   {name}: {count}")
else:
    print("   none")
print("3. Relations between nodes:", len(assertions))


In [ ]:
limit = 100
shown_claims = claims[:limit]
print(f"4. Claims ({len(shown_claims)} of {len(claims)}, limit {limit})")
for index, claim in enumerate(shown_claims, 1):
    print(
        f"{index}. [{claim.verification_status.value}/{claim.scientific_status.value}] "
        f"{claim.normalized}"
    )


In [ ]:
shown_relations = assertions[:limit]
by_iri = {item.iri: item for item in claims}
print(f"5. Relations ({len(shown_relations)} of {len(assertions)}, limit {limit})")
for index, link in enumerate(shown_relations, 1):
    subject = by_iri.get(link.subject_iri)
    obj = by_iri.get(link.object_iri)
    relation = link.relation_iri.rsplit("/", 1)[-1]
    left = subject.normalized if subject is not None else link.subject_iri
    right = obj.normalized if obj is not None else link.object_iri
    print(
        f"{index}. [{link.verification_status.value}] {left} --{relation}--> {right}"
    )


In [ ]:
question = "Why is Ethiopia exposed to El Nino?"
result = await run_research(
    ctx,
    question,
    FilterSpec(),
    rounds=settings.research_max_rounds,
    external=False,
)
print(question)
print()
print(result.answer)
print()
print("qualitative:", result.strict.qualitative)
if result.strict.quantitative_model:
    print("model:", result.strict.quantitative_model)
    print("uncertainty:", result.strict.uncertainty)
print(f"cited statements: {len(result.strict.statements)}")
for statement in result.strict.statements:
    print(f"- [{statement.scientific_status}] {statement.text}")
if result.strict.assumptions:
    print("assumptions:")
    for item in result.strict.assumptions:
        print(f"- {item}")
if result.strict.gaps:
    print("gaps:")
    for item in result.strict.gaps:
        print(f"- {item}")
if result.strict.contradictions:
    print(f"contradictions: {len(result.strict.contradictions)}")
    for item in result.strict.contradictions:
        print(f"- {item.summary}: {item.assertion_iri}")
